## Task 1: Multiple-Task Learning

**Goal:** Solve a regression problem in a 1multiple-task learning (MTL) scheme.

1. Set up reproducibility and device:

```python
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
```

2. Generate a correlated regression dataset — 8 features built as linear combinations of 8 independent base variables with noise. The target `y` depends on non-linear transformations of those features:

```python
def generate_correlated_regression_data(n_samples=3000, random_state=42):
    rng = np.random.default_rng(random_state)
    z = rng.normal(0, 1, size=(n_samples, 8))
    x1, x2, x3, x4 = z[:,0], z[:,1], z[:,2], z[:,3]
    x5, x6, x7, x8 = z[:,4], z[:,5], z[:,6], z[:,7]

    noise_small = lambda scale=0.05: rng.normal(0, scale, size=n_samples)
    noise_mid   = lambda scale=0.15: rng.normal(0, scale, size=n_samples)

    X = np.column_stack([
        x1 + x2 + noise_small(),              # 1
        2.0 * x3 - 0.5 * x4 + noise_small(),  # 2
        -1.2 * x5 + noise_small(),             # 3
        x6 + x7 + noise_small(),               # 4
        0.7 * x7 - 0.7 * x8 + noise_small(),  # 5
        1.5 * x1 - 0.8 * x3 + noise_mid(),    # 6
        x2 + x4 + x6 + noise_mid(),            # 7
        0.5 * x5 + 0.5 * x8 + noise_mid(),    # 8
    ])

    U = np.column_stack([
        X[:,0] + np.sin(X[:,1]),
        X[:,2]*X[:,3] + X[:,4]*X[:,5],
        -X[:,7]*np.sin(X[:,6]),
    ])

    y = (
        3.5 * U[:, 0]
        + 0.8 * (U[:, 1] ** 2)
        + 1.2 * np.sin(U[:, 2])
        + rng.normal(0, 0.7, size=n_samples)
    )
    return X, y
```

3. Split the data into train / val / test sets. Implement `Dataset` and `DataLoader`.

4. Implement the baseline regression model `BaseRegressionNet`:

```python
BaseRegressionNet(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
)
```

5. Implement the MTL model `MTLAutoencoderRegressor`. The encoder maps input `X` to a latent vector `z` of dimension 4. The decoder reconstructs `X` from `z`. The regressor predicts `y` from `z`. The model returns `(X_hat, y_hat)`:

```python
MTLAutoencoderRegressor(
  (encoder): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=32, out_features=4, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=32, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=64, out_features=8, bias=True)
  )
  (regressor): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=16, out_features=1, bias=True)
  )
)
```

6. Train and evaluate both models. The MTL loss is a weighted sum of reconstruction loss and regression loss:

```python
reconstruction_criterion = nn.MSELoss()
regression_criterion     = nn.MSELoss()

X_hat, y_hat = model(X_batch)
recon_loss   = reconstruction_criterion(X_hat, X_batch)
reg_loss     = regression_criterion(y_hat, y_batch)
total_loss   = alpha_recon * recon_loss + beta_reg * reg_loss
```

Use the following training settings:

```python
BATCH_SIZE   = 128
EPOCHS       = 1000
LR           = 1e-3
WEIGHT_DECAY = 1e-4

ALPHA_RECON  = 0.4   # reconstruction loss weight
BETA_REG     = 1.0   # regression loss weight
```

For datasets generated with different random seeds, compute and compare regression metrics: **MSE**, **MAE**, **R²**.

**Assignment:** Implement, train and compare regression models trained in single-task and multiple-task learning schemes.


In [1]:
import torch

from utils import set_seed, generate_correlated_regression_data

SEED = 42
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

set_seed(SEED)

In [2]:
BATCH_SIZE   = 128
EPOCHS       = 1000
LR           = 1e-3
WEIGHT_DECAY = 1e-4

ALPHA_RECON  = 0.4   # reconstruction loss weight
BETA_REG     = 1.0   # regression loss weight

In [3]:
from utils import make_loaders, generate_correlated_regression_data

X, y = generate_correlated_regression_data(n_samples=8000, random_state=SEED)
train_loader, val_loader, test_loader = make_loaders(X, y, batch_size=BATCH_SIZE)

In [4]:
from utils import BaseRegressionNet, MTLAutoencoderRegressor, train_baseline, train_mtl
import torch.nn as nn

baseModel = BaseRegressionNet().to(device)
mtlModel = MTLAutoencoderRegressor().to(device)

print("Training baseline model: ")
base_model = train_baseline(BaseRegressionNet().to(device), train_loader, val_loader, criterion=nn.MSELoss(), device=device, weight_decay=WEIGHT_DECAY, epochs=EPOCHS, lr=LR)

print("\nTraining MTL model: ")
mtl_model = train_mtl(MTLAutoencoderRegressor().to(device), train_loader, val_loader, device=device, weight_decay=WEIGHT_DECAY, epochs=EPOCHS, lr=LR, alpha_recon=ALPHA_RECON, beta_reg=BETA_REG)

Training baseline model: 
Epoch 100/1000  train=33.4151  val=22.5677
Epoch 200/1000  train=26.5177  val=16.7506
Epoch 300/1000  train=24.8869  val=14.8423
Epoch 400/1000  train=21.0679  val=13.7396
Epoch 500/1000  train=18.0511  val=11.3182
Epoch 600/1000  train=18.7223  val=12.7195
Epoch 700/1000  train=19.6222  val=11.0423
Epoch 800/1000  train=18.8280  val=12.5647
Epoch 900/1000  train=21.5074  val=12.1022
Epoch 1000/1000  train=19.7256  val=11.0928

Training MTL model: 
Epoch 100/1000  train=33.0549  val=19.5961
Epoch 200/1000  train=27.1920  val=14.9872
Epoch 300/1000  train=24.3130  val=18.3778
Epoch 400/1000  train=24.6648  val=17.6437
Epoch 500/1000  train=21.3153  val=15.8151
Epoch 600/1000  train=20.4414  val=17.0546
Epoch 700/1000  train=21.1209  val=14.2749
Epoch 800/1000  train=22.9082  val=15.4736
Epoch 900/1000  train=21.1460  val=13.7748
Epoch 1000/1000  train=20.8141  val=13.9147


In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def evaluate(model, loader, is_mtl=False):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            out = model(X_batch)
            preds = out[1] if is_mtl else out
            y_pred.extend(preds.cpu().numpy().flatten())
            y_true.extend(y_batch.numpy().flatten())
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return {
        "MSE": mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2":  r2_score(y_true, y_pred),
    }

m_base = evaluate(base_model, test_loader, is_mtl=False)
m_mtl  = evaluate(mtl_model,  test_loader, is_mtl=True)

print(f"{'Model':<20} {'MSE':>8} {'MAE':>8} {'R2':>8}")
print("-" * 46)
print(f"{'BaseRegression':<20} {m_base['MSE']:>8.4f} {m_base['MAE']:>8.4f} {m_base['R2']:>8.4f}")
print(f"{'MTL':<20} {m_mtl['MSE']:>8.4f} {m_mtl['MAE']:>8.4f} {m_mtl['R2']:>8.4f}")

Model                     MSE      MAE       R2
----------------------------------------------
BaseRegression        11.3598   1.7349   0.9275
MTL                   11.0487   1.8620   0.9295


In [6]:
SEEDS = [0, 7, 123]
results = []

for s in SEEDS:
    X_s, y_s = generate_correlated_regression_data(n_samples=8000, random_state=s)
    tr, va, te, _, _ = make_loaders(X_s, y_s, batch_size=BATCH_SIZE)

    b = train_baseline(BaseRegressionNet().to(device), tr, va, criterion=nn.MSELoss(), device=device, weight_decay=WEIGHT_DECAY, epochs=EPOCHS, lr=LR)
    m = train_mtl(MTLAutoencoderRegressor().to(device), tr, va, device=device, weight_decay=WEIGHT_DECAY, epochs=EPOCHS, lr=LR, alpha_recon=ALPHA_RECON, beta_reg=BETA_REG)

    rb = evaluate(b, te, is_mtl=False)
    rm = evaluate(m, te, is_mtl=True)
    results.append((s, rb, rm))

print(f"\n{'='*62}")
print(f"{'':6} {'--- Baseline ---':^20} {'--- MTL ---':^20}")
print(f"{'Seed':<6} {'MSE':>8} {'MAE':>8} {'R2':>8}   {'MSE':>8} {'MAE':>8} {'R2':>8}")
print(f"{'='*62}")
for s, rb, rm in results:
    print(f"{s:<6} {rb['MSE']:>8.4f} {rb['MAE']:>8.4f} {rb['R2']:>8.4f}   {rm['MSE']:>8.4f} {rm['MAE']:>8.4f} {rm['R2']:>8.4f}")
print(f"{'='*62}")

ValueError: not enough values to unpack (expected 5, got 3)